# Environment setup

Clones the repo, installs the package, runs the offline test suite and records the environment report (GPU, CUDA, versions, commit).

In [ ]:
# Havyn Stage 5 - environment setup and verification gate.
# Every notebook runs this first: no notebook trains or generates data on an
# unverified environment.
import sys, shutil, os
if not shutil.which("git"):
    raise RuntimeError("git missing")
!git clone --depth 1 https://github.com/HamzaNinjaCoder1/MCP-Packet-Tracer.git /content/MCP-Packet-Tracer 2>/dev/null || git -C /content/MCP-Packet-Tracer pull --ff-only
import sys
sys.path.insert(0, "/content/MCP-Packet-Tracer/havyn-stage5/scripts")
os.chdir("/content/MCP-Packet-Tracer")
from colab_setup import setup, save_report
env = setup(run_tests=True, use_drive=True)
assert not env.errors, f"environment verification failed: {env.errors}"
assert env.tests_passed, "offline suite failed - STOP, do not continue"
from env_report import build_report, recommend_config
report = build_report(repo="/content/MCP-Packet-Tracer", dataset_version="v0")
import json, pathlib
out_dir = pathlib.Path(env.drive_dir or "/content/havyn-artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "env-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(recommend_config(report.get("gpu", {})), indent=2))
print(json.dumps(report, indent=2)[:2000])


In [ ]:
import os
os.environ["HAVYN_DRIVE"] = env.drive_dir or "/content/havyn-artifacts"
os.environ["HAVYN_REPO"] = "/content/MCP-Packet-Tracer"
print("HAVYN_DRIVE =", os.environ["HAVYN_DRIVE"])

In [ ]:
from hf_auth import status, MISSING_TOKEN_MESSAGE

_state = status()
print("Hugging Face token:", "present" if _state.present else "absent")
print("source:", _state.source or "none")
if not _state.present:
    print(MISSING_TOKEN_MESSAGE)

**Read this before any expensive cell.** The dashboard shows GPU, VRAM, Python, commit, dataset version, model configuration, HF auth status (token hidden), the next step and a storage estimate.

In [ ]:
!python havyn-stage5/scripts/colab_dashboard.py --next "02 engine validation"

Cheap Hugging Face check: token presence (never printed), Hub metadata and a tiny tokenizer. It does not download model weights.

In [ ]:
!python havyn-stage5/scripts/hf_smoke_test.py --dry-run